# 04e. Veto de mãe

Censo sem CPF no [`04_atribuir.ipynb`](04_atribuir.ipynb) cujo **melhor par**
cai no veto de mãe: as duas mães preenchidas, nem iguais nem prefixo
(`n ≥ 2` no começo), e não é o caso de nome completo e data iguais.

O [`04c_decompor_sem_cpf.ipynb`](04c_decompor_sem_cpf.ipynb) mistura isso com
primeiro nome ausente ou com mais de 1 de Levenshtein. Aqui os dois ficam
separados. A escada já aceita Jaro-Winkler da mãe ≥ 0,75, e de 0,95 a 0,99
não aplica esses vetos. Este notebook continua mostrando o corte estrito.


In [1]:
import sys
from pathlib import Path

from IPython.display import display

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

from config import (
    SPLINK_ATRIBUICAO,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA_APLICACAO,
    TABELA_CPF_LIMPA_APLICACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

print_paths()
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')
con = get_connection()
drop_splink_temp_tables(con)
require_tables(
    con,
    [TABELA_CENSO_LIMPA_APLICACAO, TABELA_CPF_LIMPA_APLICACAO],
    notebook_origem='00b',
)
materialize_splink_input(
    con,
    censo_table=TABELA_CENSO_LIMPA_APLICACAO,
    cpf_table=TABELA_CPF_LIMPA_APLICACAO,
)


OUTPUT_DIR_BASE: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output
OUTPUT_DIR: /home/ibge.gov.br/ramon.goncalves/data/probabilistico_output/uf_21
recorte: uf_21
CPF_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf.parquet
CENSO_PESSOAS_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pessoas_2022.parquet
CENSO_PES_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_pes_nome.parquet
CPF_CPF_NOME_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/cpf/cpf_cpf_nome.parquet
CENSO_ESPECIE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_especie.parquet
CENSO_ENDERECO_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_endereco.parquet
CENSO_FACE_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_face.parquet
CENSO_LOGR_ARQUIVO: /home/ibge.gov.br/ramon.goncalves/singed/bases/bronze/censo/2022/censo_

('censo_limpo_aplicacao', 'cpf_limpo_aplicacao')

In [2]:
con.execute(f'''
CREATE OR REPLACE VIEW splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_censo,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_cpf,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE VIEW atribuicao_2 AS
SELECT unique_id_censo
FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

print(
    'Pares no parquet:',
    f"{con.execute('SELECT COUNT(*) FROM splink_predictions').fetchone()[0]:,}",
)
print(
    'Atribuições:',
    f"{con.execute('SELECT COUNT(*) FROM atribuicao_2').fetchone()[0]:,}",
)


Pares no parquet: 6,598,211
Atribuições: 3,661,469


Melhor par de quem não saiu no 04. `veto_mae` exige primeiro
nome preenchido e com no máximo 1 de Levenshtein: se o primeiro nome já
reprova, o par vai para `so_primeiro` e não entra na leitura da mãe.


In [3]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_mae = f"least(len({tok_c}), len({tok_p}))"
prefixo_mae = (
    f"{n_mae} >= 2 AND "
    f"list_slice({tok_c}, 1, {n_mae}) = list_slice({tok_p}, 1, {n_mae})"
)
# Nome completo e data iguais passam mesmo com mãe diferente.
excecao = (
    "ca.nome_completo_phon IS NOT NULL AND pb.nome_completo_phon IS NOT NULL "
    "AND ca.nome_completo_phon = pb.nome_completo_phon "
    "AND ca.data_nascimento IS NOT NULL AND pb.data_nascimento IS NOT NULL "
    "AND ca.data_nascimento = pb.data_nascimento"
)

con.execute(f'''
CREATE OR REPLACE TABLE censo_sem_cpf AS
SELECT c.unique_id
FROM {SPLINK_INPUT_VIEW} c
WHERE c.origem = 'censo'
  AND NOT EXISTS (
      SELECT 1 FROM atribuicao_2 a WHERE a.unique_id_censo = c.unique_id
  )
''')

con.execute('''
CREATE OR REPLACE TABLE melhor_par AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM splink_predictions
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY unique_id_censo
    ORDER BY match_probability DESC, unique_id_cpf
) = 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE melhor_sem AS
SELECT
    s.unique_id AS unique_id_censo,
    p.unique_id_cpf,
    p.match_probability,
    CASE
        WHEN p.unique_id_cpf IS NULL THEN 'sem_par'
        WHEN ca.primeiro_nome_phon IS NULL
             OR pb.primeiro_nome_phon IS NULL
             OR levenshtein(ca.primeiro_nome_phon, pb.primeiro_nome_phon) > 1
            THEN 'so_primeiro'
        WHEN ca.nome_mae_phon IS NOT NULL
             AND pb.nome_mae_phon IS NOT NULL
             AND ca.nome_mae_phon <> pb.nome_mae_phon
             AND NOT ({prefixo_mae})
             AND NOT ({excecao})
            THEN 'veto_mae'
        ELSE 'passaria'
    END AS porta
FROM censo_sem_cpf s
LEFT JOIN melhor_par p ON p.unique_id_censo = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = s.unique_id
LEFT JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_cpf
''')

display(con.execute('''
SELECT
    porta,
    COUNT(*) AS n,
    SUM(CASE WHEN match_probability >= 0.75 THEN 1 ELSE 0 END) AS n_p075,
    ROUND(AVG(match_probability), 3) AS avg_p
FROM melhor_sem
GROUP BY 1
ORDER BY n DESC
''').df())


,porta,n,n_p075,avg_p
0,sem_par,1463222,0.0,NaN
1,passaria,564180,260495.0,0.562
2,veto_mae,309040,209547.0,0.715
3,so_primeiro,81119,68747.0,0.847


No veto de mãe, como os dois nomes da mãe se afastam, ao lado
do nome da pessoa e da data. `mesmas_palavras_ordem` e `contem_sem_prefixo`
são o lugar em que a comparação pode estar rígida demais: as palavras batem,
a regra de prefixo não.


In [4]:
tok_c = "string_split(ca.nome_mae_phon, ' ')"
tok_p = "string_split(pb.nome_mae_phon, ' ')"
n_c = f"len({tok_c})"
n_p = f"len({tok_p})"
inter = f"len(list_intersect({tok_c}, {tok_p}))"

con.execute(f'''
CREATE OR REPLACE TABLE veto_mae AS
SELECT
    m.unique_id_censo,
    m.unique_id_cpf,
    m.match_probability,
    ca.nome_completo_phon AS nome_censo,
    pb.nome_completo_phon AS nome_cpf,
    ca.nome_mae_phon AS mae_censo,
    pb.nome_mae_phon AS mae_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) AS jw_mae,
    CASE
        WHEN list_sort({tok_c}) = list_sort({tok_p}) AND {tok_c} <> {tok_p}
            THEN 'mesmas_palavras_ordem'
        WHEN {tok_c} = {tok_p}
            THEN 'mesmas_palavras'
        WHEN {inter} = least({n_c}, {n_p}) AND {n_c} <> {n_p}
            THEN 'contem_sem_prefixo'
        WHEN jaro_winkler_similarity(ca.nome_mae_phon, pb.nome_mae_phon) >= 0.92
            THEN 'parecidas'
        WHEN {inter} >= 1
            THEN 'alguma_palavra'
        ELSE 'sem_palavra'
    END AS mae_dif,
    CASE
        WHEN ca.nome_completo_phon = pb.nome_completo_phon THEN 'exato'
        WHEN ca.primeiro_nome_phon = pb.primeiro_nome_phon
             AND ca.ultimo_nome_phon = pb.ultimo_nome_phon THEN 'pontas'
        ELSE 'outro'
    END AS nome_pessoa,
    CASE
        WHEN ca.data_nascimento IS NULL AND pb.data_nascimento IS NULL THEN 'as_duas_nulas'
        WHEN ca.data_nascimento IS NULL OR pb.data_nascimento IS NULL THEN 'uma_nula'
        WHEN ca.data_nascimento = pb.data_nascimento THEN 'exata'
        ELSE 'diferente'
    END AS data_par
FROM melhor_sem m
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
WHERE m.porta = 'veto_mae'
''')

n_excecao = con.execute('''
SELECT COUNT(*) FROM veto_mae
WHERE nome_pessoa = 'exato' AND data_par = 'exata'
''').fetchone()[0]
print('Veto de mãe:', f"{con.execute('SELECT COUNT(*) FROM veto_mae').fetchone()[0]:,}")
print('Nome e data exatos dentro do veto (tem de ser 0):', n_excecao)

display(con.execute('''
SELECT
    mae_dif,
    nome_pessoa,
    data_par,
    COUNT(*) AS n,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct,
    ROUND(AVG(match_probability), 3) AS avg_p,
    ROUND(AVG(jw_mae), 3) AS avg_jw_mae
FROM veto_mae
GROUP BY 1, 2, 3
ORDER BY n DESC
''').df())


Veto de mãe: 309,040
Nome e data exatos dentro do veto (tem de ser 0): 0


,mae_dif,nome_pessoa,data_par,n,pct,avg_p,avg_jw_mae
0,alguma_palavra,exato,diferente,45930,14.9,0.517,0.742
1,parecidas,outro,exata,38760,12.5,0.917,0.964
2,parecidas,pontas,exata,35966,11.6,0.914,0.965
3,alguma_palavra,exato,uma_nula,29208,9.5,0.501,0.760
4,parecidas,exato,diferente,28526,9.2,0.850,0.962
5,alguma_palavra,outro,exata,26569,8.6,0.879,0.841
6,parecidas,exato,uma_nula,20446,6.6,0.765,0.963
7,alguma_palavra,pontas,exata,18272,5.9,0.860,0.848
8,sem_palavra,exato,diferente,10407,3.4,0.432,0.603
9,contem_sem_prefixo,outro,exata,7190,2.3,0.837,0.878


Amostra. Primeiro as mães com as mesmas palavras ou uma dentro
da outra. Depois as parecidas com nome da pessoa exato. Por último, mãe sem
palavra em comum e nome da pessoa exato: o veto no caso claro.


In [5]:
cols = '''
    match_probability,
    mae_dif,
    nome_pessoa,
    data_par,
    ROUND(jw_mae, 3) AS jw_mae,
    nome_censo,
    nome_cpf,
    mae_censo,
    mae_cpf,
    dob_censo,
    dob_cpf
'''

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif IN ('mesmas_palavras_ordem', 'mesmas_palavras', 'contem_sem_prefixo')
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'parecidas' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 40
''').df())

display(con.execute(f'''
SELECT {cols}
FROM veto_mae
WHERE mae_dif = 'sem_palavra' AND nome_pessoa = 'exato'
ORDER BY match_probability DESC
LIMIT 20
''').df())


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.996233,contem_sem_prefixo,outro,exata,0.934,EBINOA LOPIS PINHEIRU,EBINOAN LOPIS PINHEIRU,RAIMUNDA CONCEICAO PINHEIRO,RAIMUNDA CONCEICAO LOPES PINHEIRO,1951-09-22,1951-09-22
1,0.996002,contem_sem_prefixo,outro,exata,0.777,KLEIMAR SIUVA ROXA,KLERIMAR SIUVA ROXA,FRANCISCA SILVA ROCHA,AFRA FRANCISCA SILVA ROCHA,1953-09-17,1953-09-17
2,0.995982,contem_sem_prefixo,pontas,exata,0.888,MARIA NIUDI SIUVA SOUZA,MARIA NIUDA SIUVA SOUZA,ANDRELINA SILVA SOUSA,ANDRELINA FRANCISCA SILVA SOUSA,1965-03-16,1965-03-16
3,0.995963,contem_sem_prefixo,pontas,exata,0.917,FRANSISKU BETU DAMASENU,FRANSISKU BENTU DAMASENU,MARIA DORES SOUSA DAMASCENO,MARIA DORES DAMASCENO,1966-06-07,1966-06-07
4,0.995925,contem_sem_prefixo,pontas,exata,0.805,JOAU ODOURIKU SOUZA,JOAU ODORIKU SOUZA,MARIA RAIMUNDA SOUSA,RAIMUNDA MARIA ODORICO SOUSA,1963-06-27,1963-06-27
5,0.995848,contem_sem_prefixo,outro,exata,0.912,MARIA FATIMA PORTELA KOREA,MARIA FATIMA PORTELA KOREIA,FRANCISCA TELES PORTELA,FRANCISCA LIMA TELES PORTELA,1954-07-12,1954-07-12
6,0.995655,contem_sem_prefixo,outro,exata,0.939,JEOVAR SIUVA FRANKA,JEOVA SIUVA FRANKA,TEREZINHA JESUS SILVA FRANCA,TEREZINHA SILVA FRANCA,1958-10-06,1958-10-06
7,0.995598,contem_sem_prefixo,pontas,exata,0.931,FRANSISKU VERA KOSTA,FRANSISKU VERAS KOSTA,JOANA IRENE VERA COSTA,JOANA IRENE COSTA,1969-06-27,1969-06-27
8,0.995559,contem_sem_prefixo,outro,exata,0.896,EVIMAR SIUVA FEREIRA,EUVIMAR SIUVA FEREIRA,ALCIMAR RIDRGUES SILVA FERREIRA,ALCIMAR SILVA FERREIRA,1969-04-13,1969-04-13
9,0.995521,contem_sem_prefixo,pontas,exata,0.924,MARIA FATIMA OLIVEITA SIUVA,MARIA FATIMA OLIVEIRA SIUVA,FRANCISQUINHA COSTA OLIVEIRA,FRANCISQUINHA SOARES COSTA OLIVEIRA,1956-10-22,1956-10-22


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.984569,parecidas,exato,diferente,0.993,FRANSISKU KAIU SOUZA SANTUS,FRANSISKU KAIU SOUZA SANTUS,LEIDE CARVALHO OLIVEIRA SOUSA,LEIDE CARVALHO OLIVERIA SOUSA,2014-02-02,2016-02-02
1,0.984569,parecidas,exato,diferente,0.994,RAIMUNDU GUILERMI PAULA KUNHA,RAIMUNDU GUILERMI PAULA KUNHA,MARIA GRANCAS RODRIGUES PAULA CUNHA,MARIA GRACAS RODRIGUES PAULA CUNHA,2009-11-30,2007-11-30
2,0.984569,parecidas,exato,diferente,0.952,AUDILENI BARUS SOUZA,AUDILENI BARUS SOUZA,MARIA ODILIA BARROS SILVA,MARIA ODILIA BARROS SOUSA,1994-11-29,1992-11-29
3,0.984569,parecidas,exato,diferente,0.941,UALEXI ANDREI AUKANTARA REIS,UALEXI ANDREI AUKANTARA REIS,WILMA ANJOS ALCANTRA,WYLMA ANJOS ALCANTARA,2005-11-28,2001-11-28
4,0.984569,parecidas,exato,diferente,0.984,PABLU ENRIKI GUIMARAIS NASIMENTU,PABLU ENRIKI GUIMARAIS NASIMENTU,ALAYNE RAVANE ALVES GUIMARAES,ALAINE RAVANE ALVES GUIMARAES,2015-06-15,2017-06-15
5,0.984569,parecidas,exato,diferente,0.989,ENRIKI SOUZA MASIEU,ENRIKI SOUZA MASIEU,ERINES BRAZ SOUSA,ERINEIS BRAZ SOUSA,2015-12-29,2011-12-29
6,0.984569,parecidas,exato,diferente,0.973,KEZIA SOFIA FEREIRA DAUVITA,KEZIA SOFIA FEREIRA DAUVITA,MARIA EDILENE MELONE FERREIRA,MARIA EDILENE MELONIO FERREIRA,2017-10-31,2015-10-31
7,0.984569,parecidas,exato,diferente,0.981,ELENIR TRINDADI,ELENIR TRINDADI,MARIA LUCENIR TRINDADE,MARIA LUCENI TRINDADE,2001-12-19,2008-12-19
8,0.984569,parecidas,exato,diferente,0.922,LAUAN KONSEIKAU LIMA,LAUAN KONSEIKAU LIMA,ELISANGILA CONCEICAO LIMA,ELIZANGELA CONCEICAO LIMA,2003-01-26,2013-01-26
9,0.984569,parecidas,exato,diferente,0.930,IASMIN VITORIA GAUVAU KONSEIKAU,IASMIN VITORIA GAUVAU KONSEIKAU,IRLANE TERVAL GALVAO,IRLANE ATERVAL GALVAO,2018-02-21,2015-02-21


,match_probability,mae_dif,nome_pessoa,data_par,jw_mae,nome_censo,nome_cpf,mae_censo,mae_cpf,dob_censo,dob_cpf
0,0.984569,sem_palavra,exato,diferente,0.609,TONI MARKUS SIUVA KOSTA,TONI MARKUS SIUVA KOSTA,MARIA RAIMUNDA SOUSA,ALDEIDE ROFINO SILVA,2014-05-14,2004-05-14
1,0.984569,sem_palavra,exato,diferente,0.537,UILIAN RAFAEU GOMIS MATUS,UILIAN RAFAEU GOMIS MATUS,BENEDITO JESUS MENDES MATOS,WDEANA SILVA GOMES,2017-05-08,2014-05-08
2,0.984569,sem_palavra,exato,diferente,0.547,ELISANDRA AUMEIDA SOUZA,ELISANDRA AUMEIDA SOUZA,MARIA FABIANA BORGES NASCIMENTO,ALEXSANDRA SILVA ALMEIDA,2005-01-24,2007-01-24
3,0.984569,sem_palavra,exato,diferente,0.618,LUIS RIKARDU XAVIER SIUVA,LUIS RIKARDU XAVIER SIUVA,KELLEN CRISTINA SILVA,ADRIANA SANTOS XAVIER,2012-05-17,2010-05-17
4,0.984569,sem_palavra,exato,diferente,0.551,VANDERLEI FAUKAU MESKITA,VANDERLEI FAUKAU MESKITA,MARIA CARMELIA ALMEIDA MESQUITA,DORILDA FALCAO CUNHA,1987-08-02,1989-08-02
5,0.984569,sem_palavra,exato,diferente,0.771,KAMILA MEIRELIS SALIS,KAMILA MEIRELIS SALIS,VANIA MEIRELES SALES,SONIA MARIA MEIRELLES SALLES,1987-06-30,1984-06-30
6,0.984569,sem_palavra,exato,diferente,0.604,AUZILENI PINHEIRU SIUVA,AUZILENI PINHEIRU SIUVA,RAIMUNDA GOMES ALMEIDA,LUIZA PINHEIRO SILVA,1974-09-21,1964-09-21
7,0.984569,sem_palavra,exato,diferente,0.862,MARLENI PEREIRA KOREIA,MARLENI PEREIRA KOREIA,BERNARDA PIRES VIANA,BERNERDA PEREIRA,1964-06-26,1967-06-26
8,0.984569,sem_palavra,exato,diferente,0.622,MARKUS ANTONIU MARINHU SA,MARKUS ANTONIU MARINHU SA,MARIA LUCIA PEREIRA,CRISTIANE LEITE MARINHO,1999-01-02,1997-01-02
9,0.984569,sem_palavra,exato,diferente,0.635,PAULU RIKARDU SEGUINS BARBOZA,PAULU RIKARDU SEGUINS BARBOZA,ELISABETH REGO BARBOSA,MARIA JESUS RABELO SEGUINS,2011-12-02,2001-12-02


In [6]:
con.close()
